In [ ]:
#| default_exp core

In [ ]:
#| export
"""lyca — Stateful Chat/AsyncChat for LiteRT-LM models."""

__all__ = ['Response', 'Chat', 'AsyncChat', 'get_text', 'extract_tool_calls', 'tool_schema']

In [ ]:
#| export
import atexit, asyncio, os, re
from dataclasses import dataclass, field

import litert_lm
from toolslm.funccall import get_schema
from msglm import mk_msg, mk_msgs
from fastcore.utils import *
from fastcore.meta import store_attr, delegates, patch

In [ ]:
#| Model download for tests (NOT exported)
from huggingface_hub import hf_hub_download
MODEL_PATH = hf_hub_download(
    repo_id='litert-community/gemma-4-E2B-it-litert-lm',
    filename='gemma-4-E2B-it.litertlm'
)
print(f'Model downloaded to: {MODEL_PATH}')

In [ ]:
#| export
def get_text(response: dict) -> str:
    "Extract text string from a LiteRT-LM response dict."
    for item in response.get('content', []):
        if item.get('type') == 'text': return item['text']
    return ''

In [ ]:
#| Tests for get_text — real data, no mocks
resp = {'content': [{'type': 'text', 'text': 'hello world'}]}
assert get_text(resp) == 'hello world'

resp_multi = {'content': [{'type': 'audio', 'data': '...'}, {'type': 'text', 'text': 'description'}]}
assert get_text(resp_multi) == 'description'

assert get_text({'content': []}) == ''
assert get_text({}) == ''
print('get_text tests passed')

In [ ]:
#| export
def extract_tool_calls(text: str) -> list[dict]:
    "Parse Gemma 4 <|tool_call>...<tool_call|> tokens from raw text."
    def cast(v):
        try: return int(v)
        except (ValueError, TypeError):
            try: return float(v)
            except (ValueError, TypeError): return {'true': True, 'false': False}.get(v.lower(), v.strip("'\"" ))
    return [{'name': n, 'arguments': {
                k: cast((v1 or v2).strip())
                for k, v1, v2 in re.findall(r'(\w+):(?:<\|"\|>(.*?)<\|"\|>|([^,}]*))', args)
             }}
            for n, args in re.findall(r'<\|tool_call>call:(\w+)\{(.*?)\}<tool_call\|>', text, re.DOTALL)]

In [ ]:
#| Tests for extract_tool_calls — real Gemma 4 token strings, no mocks
raw = '<|tool_call>call:search{query:<|"|>hello<|"|>,top_k:5}<tool_call|>'
tcs = extract_tool_calls(raw)
assert len(tcs) == 1
assert tcs[0]['name'] == 'search'
assert tcs[0]['arguments'] == {'query': 'hello', 'top_k': 5}

assert extract_tool_calls('plain text with no tool calls') == []

raw2 = '<|tool_call>call:a{x:1}<tool_call|> some text <|tool_call>call:b{y:<|"|>hi<|"|>}<tool_call|>'
assert len(extract_tool_calls(raw2)) == 2

raw3 = '<|tool_call>call:fn{a:42,b:3.14,c:true,d:<|"|>text<|"|>}<tool_call|>'
tc = extract_tool_calls(raw3)[0]['arguments']
assert tc == {'a': 42, 'b': 3.14, 'c': True, 'd': 'text'}
print('extract_tool_calls tests passed')

In [ ]:
#| export
def tool_schema(f) -> dict:
    "Show toolslm JSON schema for function `f` (informational — lyca passes callables directly)."
    return get_schema(f, pname='parameters')

In [ ]:
#| Tests for tool_schema — real function, no mocks
def _add(a: int, b: int) -> int:
    "Add two numbers."
    return a + b

s = tool_schema(_add)
assert s['name'] == '_add'
assert 'parameters' in s
assert 'a' in s['parameters']['properties']
assert 'b' in s['parameters']['properties']
print('tool_schema tests passed')

In [ ]:
#| export
@dataclass
class Response:
    "Return value from Chat.__call__."
    content: str
    tool_calls: list = field(default_factory=list)
    finish_reason: str = 'stop'
    model: str = ''

    def _repr_markdown_(self):
        md = self.content
        if self.tool_calls:
            md += '\n\n' + '\n'.join(f"🔧 {tc['name']}({tc.get('arguments',{})})" for tc in self.tool_calls)
        details = [f"model: `{self.model}`", f"finish_reason: `{self.finish_reason}`"]
        md += f"\n\n<details>\n\n- " + '\n- '.join(details) + '\n\n</details>'
        return md

In [ ]:
#| Tests for Response — real construction, no mocks
r = Response('hello', [], 'stop', 'gemma-4-e2b')
assert r.content == 'hello'
assert r.finish_reason == 'stop'
assert r.tool_calls == []

md = r._repr_markdown_()
assert 'hello' in md
assert 'gemma-4-e2b' in md

r2 = Response('result', [{'name': 'search', 'arguments': {'q': 'test'}}], 'tool_calls', 'gemma-4-e2b')
md2 = r2._repr_markdown_()
assert '🔧 search' in md2
print('Response tests passed')

In [ ]:
#| export
_engines: dict = {}

def _get_engine(model_path: str) -> 'litert_lm._Engine':
    "Get or create a cached Engine for `model_path`. One Engine per model."
    p = os.path.abspath(model_path)
    if p not in _engines:
        _engines[p] = litert_lm.Engine(p)
        _engines[p].__enter__()
    return _engines[p]

def _close_engines():
    "Close all cached engines."
    for e in _engines.values():
        try: e.__exit__(None, None, None)
        except Exception: pass
    _engines.clear()

atexit.register(_close_engines)

In [ ]:
#| export
class Chat:
    def __init__(self, model_path: str, sp: str = '', tools: list = None,
                 hist: list = None, max_steps: int = 5, max_tokens: int = 1024):
        "LiteRT-LM stateful chat client."
        tools = listify(tools)
        hist = mk_msgs(hist) if hist else []
        store_attr()
        self._engine = _get_engine(model_path)
        self._init_conv()
        atexit.register(self.close)

    def _init_conv(self):
        msgs = []
        if self.sp: msgs.append({'role': 'system', 'content': [{'type': 'text', 'text': self.sp}]})
        msgs += self.hist
        self._conv = self._engine.create_conversation(messages=msgs, tools=self.tools or None)
        self._conv.__enter__()

    def __call__(self, pr=None) -> Response:
        "Send `pr` to model; return Response."
        if pr is not None:
            self.hist.append({'role': 'user', 'content': [{'type': 'text', 'text': pr}]})
        resp = self._conv.send_message(pr)
        text = get_text(resp)
        tcs = extract_tool_calls(text)
        asst = {'role': 'assistant', 'content': [{'type': 'text', 'text': text}]}
        if resp.get('tool_calls'): asst['tool_calls'] = resp['tool_calls']
        if resp.get('tool_responses'): asst['tool_responses'] = resp['tool_responses']
        self.hist.append(asst)
        fr = 'tool_calls' if tcs else 'stop'
        return Response(content=text, tool_calls=tcs, finish_reason=fr, model=self.model_path)

    @property
    def h(self): return self.hist

    def reset(self):
        "Clear history and recreate conversation (keeps engine alive)."
        self._close_conv()
        self.hist = []
        self._init_conv()

    def _close_conv(self):
        try: self._conv.__exit__(None, None, None)
        except Exception: pass

    def close(self):
        "Close the conversation."
        self._close_conv()

    def __enter__(self): return self
    def __exit__(self, *exc): self.close()

In [ ]:
#| Test: basic Chat with real model
with Chat(MODEL_PATH, sp='Be terse. Reply in one sentence.') as c:
    r = c('What is 2+2?')
    assert isinstance(r, Response)
    assert isinstance(r.content, str)
    assert len(r.content) > 0
    assert r.finish_reason == 'stop'
    assert r.model == MODEL_PATH
    print(f'Basic chat response: {r.content}')

In [ ]:
#| Test: stateful multi-turn with real model
with Chat(MODEL_PATH) as c:
    c('My name is Karthik.')
    r = c("What's my name?")
    assert 'Karthik' in r.content, f'Expected Karthik in response, got: {r.content}'
    print(f'Multi-turn response: {r.content}')

In [ ]:
#| Test: history tracking
with Chat(MODEL_PATH) as c:
    c('Hello')
    assert len(c.h) == 2, f'Expected 2 history entries, got {len(c.h)}'
    assert c.h[0]['role'] == 'user'
    assert c.h[1]['role'] == 'assistant'
    print('History tracking test passed')

In [ ]:
#| Test: reset clears history
with Chat(MODEL_PATH) as c:
    c('Hello')
    assert len(c.h) == 2
    c.reset()
    assert c.h == []
    print('Reset test passed')

In [ ]:
#| Test: pre-populated history
with Chat(MODEL_PATH, hist=['What is 2+2?', '4', 'And 3+3?']) as c:
    assert len(c.hist) == 3
    r = c('And 4+4?')
    assert isinstance(r, Response)
    assert len(r.content) > 0
    print(f'Pre-populated history response: {r.content}')

In [ ]:
#| Test: tool calling with real model
def add_numbers(a: float, b: float) -> float:
    """Adds two numbers.
    Args:
        a: The first number.
        b: The second number.
    """
    return a + b

with Chat(MODEL_PATH, tools=[add_numbers]) as c:
    r = c('What is 123 + 456?')
    assert isinstance(r, Response)
    print(f'Tool calling response: {r.content}')
    # The model should compute the result via tool call
    assert '579' in r.content, f'Expected 579 in response, got: {r.content}'

In [ ]:
#| Test: engine caching
e1 = _get_engine(MODEL_PATH)
e2 = _get_engine(MODEL_PATH)
assert e1 is e2
print('Engine caching test passed')

In [ ]:
#| Test: context manager protocol
with Chat(MODEL_PATH) as c:
    r = c('Say hello')
    assert len(r.content) > 0
print('Context manager test passed')

In [ ]:
#| export
class AsyncChat(Chat):
    "Async streaming variant of Chat."

    @delegates(Chat.__init__)
    def __init__(self, **kwargs): super().__init__(**kwargs)

    async def __call__(self, pr=None):
        "Async generator: yields str chunks, then final Response."
        if pr is not None:
            self.hist.append({'role': 'user', 'content': [{'type': 'text', 'text': pr}]})
        loop = asyncio.get_event_loop()
        chunks_iter = await loop.run_in_executor(None, self._conv.send_message_async, pr)
        full = []
        for chunk in chunks_iter:
            for item in chunk.get('content', []):
                if item.get('type') == 'text':
                    t = item['text']
                    full.append(t)
                    yield t
        text = ''.join(full)
        tcs = extract_tool_calls(text)
        asst = {'role': 'assistant', 'content': [{'type': 'text', 'text': text}]}
        self.hist.append(asst)
        fr = 'tool_calls' if tcs else 'stop'
        yield Response(content=text, tool_calls=tcs, finish_reason=fr, model=self.model_path)

    async def acall(self, pr=None) -> Response:
        "Convenience: collect all chunks, return final Response."
        r = None
        async for chunk in self(pr):
            if isinstance(chunk, Response): r = chunk
        return r

    async def __aenter__(self): return self
    async def __aexit__(self, *exc): self.close()

In [ ]:
#| Test: AsyncChat streaming with real model
import asyncio

async def _test_async_stream():
    async with AsyncChat(model_path=MODEL_PATH) as c:
        chunks = []
        async for chunk in c('Say hello'):
            chunks.append(chunk)
        assert isinstance(chunks[-1], Response)
        strs = [ch for ch in chunks if isinstance(ch, str)]
        assert len(strs) > 0
        assert chunks[-1].content == ''.join(strs)
        print(f'Async stream response: {chunks[-1].content}')

await _test_async_stream()

In [ ]:
#| Test: AsyncChat acall with real model
async def _test_acall():
    async with AsyncChat(model_path=MODEL_PATH) as c:
        r = await c.acall('What is Python?')
        assert isinstance(r, Response)
        assert len(r.content) > 0
        print(f'Async acall response: {r.content[:100]}')

await _test_acall()